In [3]:

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv


In [4]:
# ============================================================
# FAKE NEWS DETECTION PROJECT (COMPLETELY FIXED KAGGLE CODE)
# Realistic Training Pipeline - No Bias, No Overfitting
# ============================================================

# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
print("✅ Step 1: Libraries Loaded!")

# ============================================================
# STEP 2 & 3: LOAD DATASETS
# ============================================================
fake_path = "/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv"
real_path = "/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv"

fake_df = pd.read_csv(fake_path)
real_df = pd.read_csv(real_path)
print(f"✅ Step 3: Loaded {fake_df.shape[0]} Fake and {real_df.shape[0]} Real articles.")

# Add Target Labels
fake_df["label"] = 0
real_df["label"] = 1

# Combine
df = pd.concat([fake_df, real_df], ignore_index=True)

# Combine Title and Text to create full content
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

# ============================================================
# STEP 7: UNBIASED CLEANING FUNCTION (The Main Fix)
# ============================================================
def clean_text_unbiased(text):
    text = str(text).lower()
    
    # 🎯 FIX 1: Remove Reuters and other publisher patterns completely.
    # This removes text like "washington (reuters) -", "reuters", "breitbart" etc.
    text = re.sub(r'^[a-z\s]+ \([a-z\s]+\) - ', '', text)
    text = re.sub(r'\([a-z\s]+\)', '', text)
    text = re.sub(r'reuters', '', text)
    
    # Remove URLs, links, html marks
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove Special Characters and Numbers (keeping only pure text words)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove Extra Spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("🔄 Step 7: Running unbiased text preprocessing to drop corporate signatures...")
df["content"] = df["content"].apply(clean_text_unbiased)

# Drop rows that became empty after tough cleaning
df = df[df["content"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)
print("✅ Step 7: Unbiased cleaning complete.")

# ============================================================
# STEP 9: TRAIN TEST SPLIT
# ============================================================
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintains realistic balanced ratio
)

# ============================================================
# STEP 10: OPTIMIZED TF-IDF VECTORIZATION
# ============================================================
# Limiting features to general words, avoiding overfitting on exact sentences
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=4000, 
    ngram_range=(1,1),  # Using unigrams prevents overfitting on unique phrases
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print("✅ Step 10: Vectorization completed.")

# ============================================================
# STEP 11: TRAIN REGULARIZED MODEL
# ============================================================
# Added strong regularization (C=0.1) so the model generalizes well 
# instead of memorizing the training dataset perfectly.
model = LinearSVC(C=0.1, max_iter=3000, random_state=42)
model.fit(X_train_tfidf, y_train)
print("✅ Step 11: Robust LinearSVC Model Trained!")

# ============================================================
# STEP 12 & 14: REALISTIC EVALUATION
# ============================================================
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*50)
print(f"🎯 REALISTIC TEST ACCURACY: {accuracy*100:.2f}%")
print("="*50)
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Fake News", "Real News"]))

# ============================================================
# STEP 17: SAVE GENERALIZED MODEL
# ============================================================
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("\n💾 model.pkl and vectorizer.pkl generated cleanly!")

# ============================================================
# STEP 18: TEST PREDICTIONS WITH RAW SENTENCES
# ============================================================
def test_prediction(sentence):
    cleaned = clean_text_unbiased(sentence)
    vectorized = vectorizer.transform([cleaned])
    pred = model.predict(vectorized)[0]
    return "REAL NEWS" if pred == 1 else "FAKE NEWS"

print("\n🔍 Live Tests with Neutral Sentences (No Reuters metadata):")
print("-" * 60)
print("Sentence 1: 'The local government announced new tax updates for small businesses starting next month.'")
print(f"👉 Prediction: {test_prediction('The local government announced new tax updates for small businesses starting next month.')}")
print("\nSentence 2: 'Shocking alert! Aliens landed in New York subway tunnels and captured citizens hide your kids!'")
print(f"👉 Prediction: {test_prediction('Shocking alert! Aliens landed in New York subway tunnels and captured citizens hide your kids!')}")
print("-" * 60)

✅ Step 1: Libraries Loaded!
✅ Step 3: Loaded 23481 Fake and 21417 Real articles.
🔄 Step 7: Running unbiased text preprocessing to drop corporate signatures...
✅ Step 7: Unbiased cleaning complete.
✅ Step 10: Vectorization completed.
✅ Step 11: Robust LinearSVC Model Trained!

🎯 REALISTIC TEST ACCURACY: 98.88%

📊 Classification Report:

              precision    recall  f1-score   support

   Fake News       0.99      0.99      0.99      4695
   Real News       0.98      0.99      0.99      4283

    accuracy                           0.99      8978
   macro avg       0.99      0.99      0.99      8978
weighted avg       0.99      0.99      0.99      8978


💾 model.pkl and vectorizer.pkl generated cleanly!

🔍 Live Tests with Neutral Sentences (No Reuters metadata):
------------------------------------------------------------
Sentence 1: 'The local government announced new tax updates for small businesses starting next month.'
👉 Prediction: REAL NEWS

Sentence 2: 'Shocking alert! Aliens